<a href="https://colab.research.google.com/github/gph05010/Medication-Info-Text-Classification/blob/main/03_%EB%B2%A0%EC%9D%B4%EC%8A%A4%EB%9D%BC%EC%9D%B8_%EB%AA%A8%EB%8D%B8_%EB%AC%B8%EC%9E%A5_%EA%B8%B0%EC%A4%80_%EB%B6%84%EB%A6%AC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

import matplotlib.pyplot as plt

In [ ]:
# sample_data = {
#     "text": [
#         "이 약은 속쓰림과 위산과다에 사용합니다.",
#         "성인은 1회 1정, 1일 3회 복용합니다.",
#         "임부는 복용 전 의사 또는 약사와 상의하십시오.",
#         "테트라사이클린계 항생제와 함께 복용하지 마십시오.",
#         "발진, 가려움 등이 나타나는 경우 복용을 중지하십시오.",
#         "습기와 빛을 피해 실온에서 보관하십시오.",
#         "이 약은 기침과 가래 증상 완화에 사용합니다.",
#         "어린이는 보호자의 지도하에 복용하십시오.",
#         "드물게 구역, 구토, 설사 등이 나타날 수 있습니다.",
#         "어린이의 손이 닿지 않는 곳에 보관하십시오."
#     ],
#     "label": [
#         "효능",
#         "복용법",
#         "주의사항",
#         "상호작용",
#         "부작용",
#         "보관법",
#         "효능",
#         "주의사항",
#         "부작용",
#         "보관법"
#     ]
# }

# df_model = pd.DataFrame(sample_data)

# df_model.head()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content/drive/MyDrive/해커톤

In [ ]:
# 문장 기준 분리 데이터셋
df_model = pd.read_csv("./medicine_sentence_label_dedup.csv")

df_model = df_model[["text", "label"]].dropna()
df_model["text"] = df_model["text"].astype(str)
df_model["label"] = df_model["label"].astype(str)

In [ ]:
# 문제 데이터, 정답 데이터, 학습 데이터, 테스트 데이터 분리
X = df_model["text"]
y = df_model["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("train size:", len(X_train))
print("test size:", len(X_test))

In [ ]:
# 모델 학습 및 평가용 함수 정의
def evaluate_model(model_name, model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)

    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        y_test,
        y_pred,
        average="macro",
        zero_division=0
    )

    precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(
        y_test,
        y_pred,
        average="weighted",
        zero_division=0
    )

    print("=" * 80)
    print(model_name)
    print("=" * 80)
    print("Accuracy:", accuracy)
    print("Macro F1:", f1_macro)
    print("Weighted F1:", f1_weighted)
    print()
    print(classification_report(y_test, y_pred, zero_division=0))

    result = {
        "model": model_name,
        "accuracy": accuracy,
        "macro_precision": precision_macro,
        "macro_recall": recall_macro,
        "macro_f1": f1_macro,
        "weighted_precision": precision_weighted,
        "weighted_recall": recall_weighted,
        "weighted_f1": f1_weighted
    }

    return result, y_pred

In [ ]:
# TF-IDF + Logistic Regression
tfidf_logreg = Pipeline([
    ("vectorizer", TfidfVectorizer(
        max_features=10000,
        ngram_range=(1, 2)
    )),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

result_tfidf_logreg, pred_tfidf_logreg = evaluate_model(
    "TF-IDF + Logistic Regression",
    tfidf_logreg,
    X_train,
    X_test,
    y_train,
    y_test
)

In [ ]:
# TF-IDF + LinearSVM
tfidf_svm = Pipeline([
    ("vectorizer", TfidfVectorizer(
        max_features=10000,
        ngram_range=(1, 2)
    )),
    ("classifier", LinearSVC(
        random_state=42
    ))
])

result_tfidf_svm, pred_tfidf_svm = evaluate_model(
    "TF-IDF + LinearSVM",
    tfidf_svm,
    X_train,
    X_test,
    y_train,
    y_test
)

In [ ]:
# TF-IDF + Multinomial Naive Bayes
tfidf_nb = Pipeline([
    ("vectorizer", TfidfVectorizer(
        max_features=10000,
        ngram_range=(1, 2)
    )),
    ("classifier", MultinomialNB())
])

result_tfidf_nb, pred_tfidf_nb = evaluate_model(
    "TF-IDF + Multinomial Naive Bayes",
    tfidf_nb,
    X_train,
    X_test,
    y_train,
    y_test
)

In [ ]:
# CountVectorizer + Logistic Regression
count_logreg = Pipeline([
    ("vectorizer", CountVectorizer(
        max_features=10000,
        ngram_range=(1, 2)
    )),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

result_count_logreg, pred_count_logreg = evaluate_model(
    "CountVectorizer + Logistic Regression",
    count_logreg,
    X_train,
    X_test,
    y_train,
    y_test
)

In [ ]:
results = [
    result_tfidf_logreg,
    result_tfidf_svm,
    result_tfidf_nb,
    result_count_logreg
]

result_df = pd.DataFrame(results)

result_df = result_df.sort_values(by="macro_f1", ascending=False)

result_df

In [ ]:
# 모델 성능 비교용 그래프
plot_df = result_df[["model", "accuracy", "macro_f1", "weighted_f1"]].set_index("model")

ax = plot_df.plot(kind="bar", figsize=(11, 5))

plt.title("Baseline Model Performance Comparison")
plt.xlabel("Model")
plt.ylabel("Score")
plt.ylim(0.90, 1.00)
plt.xticks(rotation=30, ha="right")
plt.legend(loc="lower right")

# 막대 위에 값 표시
for container in ax.containers:
    ax.bar_label(container, fmt="%.3f", fontsize=8, padding=3)

plt.tight_layout()
plt.show()

In [ ]:
# 모델 성능 비교용 가로 막대 그래프
plot_df = result_df[["model", "accuracy", "macro_f1", "weighted_f1"]].set_index("model")

# 보기 좋은 순서: 성능 좋은 모델이 위에 오도록 정렬
plot_df = plot_df.sort_values("macro_f1", ascending=True)

ax = plot_df.plot(
    kind="barh",
    figsize=(10, 6)
)

plt.title("Baseline Model Performance Comparison")
plt.xlabel("Score")
plt.ylabel("Model")
plt.xlim(0.90, 1.00)
plt.legend(loc="lower right")

# 막대 끝에 값 표시
for container in ax.containers:
    ax.bar_label(
        container,
        fmt="%.3f",
        fontsize=8,
        padding=3
    )

plt.tight_layout()
plt.show()

전체적으로 잘 맞추는 것으로 나타났으며, 가장 성능이 높은 것은 TF-IDF + LinearSVM이다.

macro_f1 score가 낮은 것으로 보아, 데이터 수가 적은 '보관법'에 대하여 정확도가 떨어지고 있을 것이다.

In [ ]:
# Colab 한글 폰트 설치
!apt-get -qq install fonts-nanum

import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import numpy as np

font_path = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
fm.fontManager.addfont(font_path)

plt.rc("font", family="NanumGothic")
plt.rcParams["axes.unicode_minus"] = False

In [ ]:
# 혼동행렬 그리기
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import numpy as np

labels = ["보관법", "복용법", "부작용", "상호작용", "주의사항", "효능"]

cm = confusion_matrix(y_test, pred_tfidf_svm, labels=labels)

# 행 기준 정규화: 실제 라벨별로 몇 %가 어디로 예측됐는지
cm_norm = cm.astype("float") / cm.sum(axis=1)[:, np.newaxis]

plt.figure(figsize=(9, 7))
im = plt.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)

plt.title("Normalized Confusion Matrix - TF-IDF + LinearSVM", fontsize=15, pad=15)
plt.xlabel("Predicted Label", fontsize=12)
plt.ylabel("True Label", fontsize=12)

plt.xticks(
    ticks=np.arange(len(labels)),
    labels=labels,
    rotation=45,
    ha="right",
    fontsize=11
)
plt.yticks(
    ticks=np.arange(len(labels)),
    labels=labels,
    fontsize=11
)

plt.colorbar(im, fraction=0.046, pad=0.04)

for i in range(len(labels)):
    for j in range(len(labels)):
        value = cm_norm[i, j]
        color = "white" if value > 0.5 else "black"
        plt.text(
            j,
            i,
            f"{value:.2f}",
            ha="center",
            va="center",
            color=color,
            fontsize=10
        )

plt.tight_layout()
plt.show()

혼동행렬을 읽어보면, '주의사항'으로 예측한 결과가 가장 많이 틀린 것을 확인할 수 있다.

'보관법'을 '주의사항'으로 잘못 예측하는 경우가 많았으며, '복용법', '부작용', '상호작용'을 '주의사항'으로 잘못 예측하는 경우도 있다.

'보관법'을 '복용법'으로 잘못 예측하는 경우도 존재한다.

In [ ]:
# 틀린 것 확인
error_df = pd.DataFrame({
    "text": X_test.values,
    "true_label": y_test.values,
    "pred_label": pred_tfidf_svm
})

error_df = error_df[error_df["true_label"] != error_df["pred_label"]]

error_df.head(20)

## 오분류 사례 분석

TF-IDF + LinearSVM 모델의 오분류 사례를 확인한 결과, 모델은 주로 문장 안에 포함된 특정 표현에 강하게 반응해 라벨을 예측하는 경향을 보였다. 전체 성능은 높았지만, `복용하지 마십시오`, `주의하십시오`, `상의하십시오`, `투여하지 마십시오`처럼 금지·주의·상담을 나타내는 표현이 들어간 문장에서 라벨 간 혼동이 발생했다.

### 1. 다른 라벨을 주의사항으로 예측하는 경향

오분류 사례에서 가장 자주 보이는 패턴은 실제 라벨이 `복용법`, `상호작용`, `보관법`, `부작용`인데 모델이 이를 `주의사항`으로 예측한 경우였다.

예를 들어 다음 문장들은 실제 라벨은 다르지만, 표현상으로는 주의사항과 매우 유사하다.

| 문장 | 실제 라벨 | 예측 라벨 |
|---|---|---|
| 권장용량을 초과하여 복용하지 마십시오. | 복용법 | 주의사항 |
| 알코올이 함유된 음료와 복용하지 마십시오. | 상호작용 | 주의사항 |
| 사용기한이 초과된 것은 복용하지 마십시오. | 보관법 | 주의사항 |
| 1개월 정도 복용하여도 증상의 개선이 없을 경우 복용을 즉각 중지하고 의사 또는 약사와 상의하십시오. | 부작용 | 주의사항 |

이 문장들은 원본 데이터에서는 각각 복용법, 상호작용, 보관법, 부작용 컬럼에 포함되어 있었지만, 문장만 따로 보면 모두 “하지 말라”, “중지하라”, “상의하라”는 주의 표현을 포함하고 있다. 따라서 모델이 해당 문장들을 `주의사항`으로 예측한 것은 단순한 오류라기보다 라벨 간 표현이 겹친 결과로 볼 수 있다.

### 2. 복용법과 주의사항의 혼동

`복용법` 라벨에서 `주의사항`으로 잘못 예측된 사례가 확인되었다.

예시:

| 문장 | 실제 라벨 | 예측 라벨 |
|---|---|---|
| 권장용량을 초과하여 복용하지 마십시오. | 복용법 | 주의사항 |
| 이 약은 가능한 최단기간동안 최소 유효용량으로 복용하십시오. | 복용법 | 주의사항 |
| 4주 이상 투여하지 마십시오. | 복용법 | 주의사항 |
| 기타 자세한 사항은 허가사항 상세정보를 확인하십시오. | 복용법 | 주의사항 |

이 문장들은 복용량, 복용 기간, 복용 조건과 관련되어 있어 실제 라벨은 `복용법`이다. 하지만 “초과하여 복용하지 마십시오”, “투여하지 마십시오”, “확인하십시오” 같은 표현이 포함되어 있어 모델은 이를 주의사항으로 판단한 것으로 보인다.

즉, 복용법 안에서도 단순한 용량 설명이 아니라 제한이나 금지를 나타내는 문장은 `주의사항`과 의미적으로 가까워져 오분류가 발생했다.

### 3. 상호작용과 주의사항의 혼동

`상호작용` 라벨도 `주의사항`으로 잘못 예측되는 사례가 있었다.

예시:

| 문장 | 실제 라벨 | 예측 라벨 |
|---|---|---|
| 약은 흡착성이 있어 타 약물과 함께 복용할 경우 흡수율이나 흡수시간에 영향을 미칠 수 있습니다. | 상호작용 | 주의사항 |
| 알코올이 함유된 음료와 복용하지 마십시오. | 상호작용 | 주의사항 |
| 복용 중 음주하지 마십시오. | 상호작용 | 주의사항 |
| 프로필렌글리콜 성분에 과민증 환자 및 알레르기 반응 경험자는 의사 또는 약사와 상의하십시오. | 상호작용 | 주의사항 |

상호작용은 특정 약물, 성분, 알코올, 다른 점안제 등과 함께 사용할 때 발생할 수 있는 문제를 다룬다. 하지만 문장 표현은 “복용하지 마십시오”, “상의하십시오”처럼 주의사항에서 자주 등장하는 표현과 겹친다.

따라서 모델이 상호작용의 핵심인 “다른 약물과 함께 사용”, “알코올”, “동시 사용”보다 금지·상담 표현에 더 강하게 반응해 `주의사항`으로 예측한 것으로 해석할 수 있다.

### 4. 보관법과 주의사항의 혼동

`보관법` 문장 중 일부도 `주의사항`으로 예측되었다.

예시:

| 문장 | 실제 라벨 | 예측 라벨 |
|---|---|---|
| 사용기한이 초과된 것은 복용하지 마십시오. | 보관법 | 주의사항 |
| 사용하지 않을 경우 뚜껑을 잘 닫아두십시오. | 보관법 | 주의사항 |
| 가연성이므로 열과 화기를 멀리해야 합니다. | 보관법 | 복용법 |

보관법 라벨은 보통 “실온에서 보관하십시오”, “습기와 빛을 피해 보관하십시오”처럼 명확한 보관 표현을 포함한다. 하지만 일부 문장은 “복용하지 마십시오”, “닫아두십시오”, “멀리해야 합니다”처럼 주의나 금지에 가까운 표현을 포함한다.

특히 `사용기한이 초과된 것은 복용하지 마십시오.`는 원본에서는 보관법에 포함되어 있지만, 문장만 보면 주의사항으로 해석해도 자연스럽다. 이는 문장 단위 분리로 인해 원래 컬럼의 맥락이 줄어든 사례라고 볼 수 있다.

### 5. 문장 단위 분리로 인한 맥락 손실

이번 오류 사례 중 일부는 문장 단위로 분리했기 때문에 발생한 것으로 보인다.

예를 들어 `다른 점안제를 동시에 사용할 경우에는 적어도 5분의 간격을 두고 투여하며...` 같은 문장은 원본 컬럼에서는 주의사항에 포함되어 있었지만, 문장만 보면 다른 약물과의 사용 간격을 설명하므로 `상호작용`처럼 보일 수 있다.

또한 `이 약의 사용과 섬광도 조사는 1~2주 간격을 두십시오.`는 원본에서는 주의사항이지만, 문장 안의 `간격`, `두십시오` 같은 표현 때문에 복용법으로 예측되었다.

이처럼 문장 단위 분리는 입력 길이를 줄이고 오류 분석을 쉽게 만드는 장점이 있지만, 앞뒤 문맥이나 원본 컬럼의 정보가 사라져 라벨 경계가 모호해질 수 있다는 한계가 있다.

### 6. 정리

오분류 사례를 종합하면 다음과 같은 특징이 확인되었다.

1. 모델은 `복용하지 마십시오`, `주의하십시오`, `상의하십시오`, `투여하지 마십시오` 같은 표현이 포함된 문장을 `주의사항`으로 예측하는 경향이 있었다.
2. `복용법`, `상호작용`, `보관법`, `부작용` 일부 문장은 금지·주의 표현을 포함해 `주의사항`과 혼동되었다.
3. 일부 문장은 사람이 보아도 하나의 라벨로만 분류하기 어려웠다.
4. 문장 단위 분리로 인해 원본 컬럼의 맥락이 사라지면서 라벨 경계가 더 모호해지는 사례가 있었다.
5. 따라서 향후 개선 방향으로는 컬럼 단위 입력과 문장 단위 입력의 성능 비교, 앞뒤 문맥을 함께 사용하는 방식, 또는 하나의 문장에 여러 라벨을 허용하는 복수 라벨 분류 방식을 검토할 수 있다.